# Assignment 1 — NumPy Vectorised Computation
**Constraint:** No element-wise Python `for` loop over a NumPy array except the loop explicitly required in Q8.

Run in Google Colab with **Runtime → Restart and run all**.

In [ ]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
import time
np.random.seed(42)
print("NumPy:", np.__version__)

## Q1

In [ ]:
A = np.random.normal(50, 12, size=(1000, 5))
print("shape:", A.shape)
print("dtype:", A.dtype)
print("memory bytes:", A.nbytes)
A = A.astype(np.float32)
print("float32 memory bytes:", A.nbytes)

## Q2

In [ ]:
A_standard = (A - A.mean(axis=0)) / A.std(axis=0)
print("means:", np.round(A_standard.mean(axis=0), 6))
print("stds: ", np.round(A_standard.std(axis=0), 6))

## Q3

In [ ]:
def minmax(X, axis):
    X = np.asarray(X, dtype=float)
    lo = X.min(axis=axis, keepdims=True)
    hi = X.max(axis=axis, keepdims=True)
    span = hi - lo
    return np.divide(X - lo, span, out=np.zeros_like(X), where=span != 0)

s0 = minmax(A, 0)
s1 = minmax(A, 1)
print("axis 0 min/max:", s0.min(0), s0.max(0))
print("axis 1 first 5 min/max:", s1.min(1)[:5], s1.max(1)[:5])
assert np.allclose(s0.min(0), 0) and np.allclose(s0.max(0), 1)
assert np.allclose(s1.min(1), 0) and np.allclose(s1.max(1), 1)

## Q4

In [ ]:
rng = np.random.default_rng(42)
mask = rng.random(A.shape) < 0.05
A[mask] = np.nan
means = np.nanmean(A, axis=0)
A = np.where(np.isnan(A), means, A)
print("remaining NaN:", np.isnan(A).sum())
assert np.isnan(A).sum() == 0

## Q5

In [ ]:
mu = A.mean(0)
sd = A.std(0)
outliers = np.abs(A - mu) > 3 * sd
med = np.median(A, axis=0)
replaced = outliers.sum(0)
A = np.where(outliers, med, A)
print("replaced per column:", replaced)

## Q6

In [ ]:
X200 = A[:200]
D = np.sqrt(((X200[:, None, :] - X200[None, :, :]) ** 2).sum(axis=2))
print("shape:", D.shape)
print("diagonal zero:", np.allclose(np.diag(D), 0))
print("symmetric:", np.allclose(D, D.T))
assert np.allclose(np.diag(D), 0) and np.allclose(D, D.T)

## Q7
`argpartition` finds the five smallest distances without fully sorting each 200-element row, which avoids unnecessary work.

In [ ]:
# Exclude each row itself by setting the diagonal to infinity, then take the five smallest distances.
D_for_neighbors = D.copy()
np.fill_diagonal(D_for_neighbors, np.inf)
nearest_idx = np.argpartition(D_for_neighbors, kth=4, axis=1)[:, :5].astype(np.int64)
print("shape:", nearest_idx.shape)
print(nearest_idx[:5])
assert nearest_idx.shape == (200, 5)
assert not np.any(nearest_idx == np.arange(200)[:, None])


## Q8

In [ ]:
values = np.random.default_rng(42).integers(0, 100, size=1_000_000)

start = time.perf_counter()
counts_vectorized = np.bincount(values, minlength=100)
vectorized_time = time.perf_counter() - start

start = time.perf_counter()
counts_loop = [0] * 100
for value in values.tolist():
    counts_loop[value] += 1
loop_time = time.perf_counter() - start

counts_loop = np.asarray(counts_loop)
speedup = loop_time / vectorized_time
print("counts equal:", np.array_equal(counts_vectorized, counts_loop))
print(f"bincount: {vectorized_time:.6f}s")
print(f"loop: {loop_time:.6f}s")
print(f"speed-up: {speedup:.2f}x")

## Q9

In [ ]:
x = np.random.default_rng(42).normal(size=10_000)
windows = sliding_window_view(x, 10)
means = windows.mean(axis=1)
rolling = pd.Series(x).rolling(10).mean().dropna().to_numpy()
print("window shape:", windows.shape)
print("matches pandas:", np.allclose(means, rolling))
assert windows.shape == (9991, 10) and np.allclose(means, rolling)

## Q10

In [ ]:
arr = np.random.default_rng(42).normal(size=(12, 8))
result = arr.reshape(4, 3, 8).mean(axis=1)
print("shape:", result.shape)
print("The first dimension represents 4 groups formed from the 12 original rows.")
print("The second dimension is the original 8-feature axis after averaging each group of 3 rows.")

## Q11

In [ ]:
np.save("clean.npy", A)
loaded = np.load("clean.npy")
print("identical:", np.array_equal(loaded, A))
assert np.array_equal(loaded, A)